# WanderWise Module 1: User Preference to Place Filtering

This notebook demonstrates:
1. Taking user preferences as natural language input
2. Simulating NLC classification (8 categories: adventure, beaches, food, historical, nature, nightlife, religious, shopping)
3. Filtering places from `goa_tourist_attractions.csv` based on classified interests
4. Visualizing extracted places on an interactive map

**Note:** The actual NLC model runs in a hosted environment. This notebook simulates the classification for demonstration.

In [ ]:
# Installation and Imports
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster
import re
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

# Try to import plotly for interactive visualization
try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("Plotly not available. Install with: pip install plotly")

print("✓ All libraries loaded successfully")

## 1. Load Goa Tourist Attractions Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('goa_tourist_attractions.csv')

print(f"Loaded {len(df)} places from dataset")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few places:")
df.head()

In [ ]:
# Explore data statistics
print("Dataset Statistics:")
print(f"Total places: {len(df)}")
print(f"Average rating: {df['rating'].mean():.2f}")
print(f"Rating range: {df['rating'].min():.1f} - {df['rating'].max():.1f}")
print(f"\nLatitude range: {df['latitude'].min():.4f} to {df['latitude'].max():.4f}")
print(f"Longitude range: {df['longitude'].min():.4f} to {df['longitude'].max():.4f}")

# Analyze place types
print("\nCommon keywords in place types:")
all_types = ' '.join(df['types'].dropna()).lower()
for keyword in ['beach', 'fort', 'church', 'temple', 'waterfall', 'museum', 'park']:
    count = all_types.count(keyword)
    if count > 0:
        print(f"  {keyword}: {count}")

## 2. Define Category Mapping

Map place names/types to interest categories based on the NLC model labels:
- **Adventure**: Forts, trekking, water sports
- **Beaches**: Beach-related attractions
- **Food**: Markets, local cuisine spots
- **Historical**: Forts, museums, old structures
- **Nature**: Waterfalls, parks, wildlife
- **Nightlife**: Party spots, clubs
- **Religious**: Temples, churches, mosques
- **Shopping**: Markets, shopping areas

In [ ]:
# Category keywords mapping
CATEGORY_KEYWORDS = {
    'adventure': [
        'fort', 'trek', 'safari', 'jeep', 'adventure', 'zip', 'paragliding',
        'water sports', 'scuba', 'diving', 'kayak', 'bungee'
    ],
    'beaches': [
        'beach', 'shore', 'coast', 'sand', 'jetty', 'seaside'
    ],
    'food': [
        'market', 'food', 'cuisine', 'restaurant', 'cafe', 'shack',
        'spice', 'plantation', 'tasting'
    ],
    'historical': [
        'fort', 'museum', 'heritage', 'historical', 'old', 'ancient',
        'colonial', 'arch', 'gateway', 'ruins', 'palace', 'cathedral',
        'basilica', 'se cathedral', 'tower', 'archaeological'
    ],
    'nature': [
        'waterfall', 'falls', 'wildlife', 'sanctuary', 'nature', 'park',
        'garden', 'spring', 'dam', 'forest', 'bird', 'butterfly',
        'plantation', 'mangrove'
    ],
    'nightlife': [
        'club', 'nightlife', 'party', 'disco', 'bar', 'lounge', 'casino'
    ],
    'religious': [
        'temple', 'church', 'basilica', 'cathedral', 'chapel', 'mosque',
        'shrine', 'monastery', 'convent', 'mahadeva', 'shri', 'bom jesus'
    ],
    'shopping': [
        'market', 'shopping', 'bazaar', 'mall', 'store', 'flea market'
    ]
}

def classify_place(place_name: str, place_types: str, address: str) -> List[str]:
    """
    Classify a place into one or more categories based on keywords.
    
    Args:
        place_name: Name of the place
        place_types: Types/tags of the place
        address: Address of the place
    
    Returns:
        List of matching categories
    """
    # Combine all text for matching
    text = f"{place_name} {place_types} {address}".lower()
    
    categories = []
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(keyword in text for keyword in keywords):
            categories.append(category)
    
    # If no category matched, mark as 'other'
    if not categories:
        categories = ['other']
    
    return categories

# Apply classification to all places
df['categories'] = df.apply(
    lambda row: classify_place(row['name'], str(row['types']), row['address']),
    axis=1
)

print("Sample classifications:")
for idx in range(min(10, len(df))):
    print(f"{df.iloc[idx]['name'][:40]:40s} -> {', '.join(df.iloc[idx]['categories'])}")

In [ ]:
# Analyze category distribution
from collections import Counter

all_categories = [cat for cats in df['categories'] for cat in cats]
category_counts = Counter(all_categories)

print("\nCategory Distribution:")
for category, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {category:15s}: {count:3d} places ({count/len(df)*100:.1f}%)")

## 3. User Preference Input and Simulation

In production, the NLC model would classify the user's natural language input.
Here, we simulate the classification output.

In [ ]:
# Simulated NLC classification function
def simulate_nlc_classification(user_input: str) -> Dict[str, bool]:
    """
    Simulate NLC model classification based on keyword matching.
    In production, this would call the hosted NLC model.
    
    Args:
        user_input: Natural language preference text
    
    Returns:
        Dictionary with category: positive/negative classification
    """
    text = user_input.lower()
    
    # Positive indicators
    positive_words = ['love', 'enjoy', 'like', 'interested', 'want', 'prefer', 'looking for']
    # Negative indicators
    negative_words = ['hate', 'dislike', 'not interested', 'avoid', 'boring', 'no ']
    
    results = {}
    
    for category, keywords in CATEGORY_KEYWORDS.items():
        # Check if category is mentioned
        category_mentioned = any(keyword in text for keyword in keywords)
        
        if category_mentioned:
            # Determine sentiment (simple rule-based)
            # Check if negative words appear near category keywords
            is_negative = False
            for neg_word in negative_words:
                if neg_word in text:
                    # Check if category keyword appears after negative word within 50 chars
                    neg_pos = text.find(neg_word)
                    for keyword in keywords:
                        if keyword in text:
                            kw_pos = text.find(keyword)
                            if 0 <= kw_pos - neg_pos <= 50:
                                is_negative = True
                                break
            
            results[category] = not is_negative  # True = positive, False = negative
    
    return results

# Test examples
test_inputs = [
    "I love beaches and historical sites, but not interested in nightlife",
    "Looking for adventure activities and nature spots",
    "Interested in religious places and local food markets",
    "I enjoy shopping and trying local cuisine, no interest in temples"
]

print("Test Classifications:\n")
for test_input in test_inputs:
    classification = simulate_nlc_classification(test_input)
    print(f"Input: \"{test_input}\"")
    print(f"Positive interests: {[k for k, v in classification.items() if v]}")
    print(f"Negative interests: {[k for k, v in classification.items() if not v]}")
    print()

## 4. Filter Places Based on User Preferences

In [ ]:
# USER INPUT - Modify this cell to test different preferences
USER_PREFERENCE = "I love exploring historical forts and beaches, interested in nature waterfalls, but not interested in nightlife or shopping"

print(f"User Input: \"{USER_PREFERENCE}\"\n")

# Classify user preferences
user_interests = simulate_nlc_classification(USER_PREFERENCE)

print("Classified Interests:")
positive_interests = [k for k, v in user_interests.items() if v]
negative_interests = [k for k, v in user_interests.items() if not v]

print(f"  Positive: {positive_interests}")
print(f"  Negative: {negative_interests}")
print()

In [ ]:
def filter_places_by_interests(
    df: pd.DataFrame,
    positive_interests: List[str],
    negative_interests: List[str],
    min_rating: float = 4.0
) -> pd.DataFrame:
    """
    Filter places based on user interests.
    
    Args:
        df: DataFrame with places
        positive_interests: Categories user is interested in
        negative_interests: Categories user wants to avoid
        min_rating: Minimum rating threshold
    
    Returns:
        Filtered DataFrame
    """
    if not positive_interests:
        print("No positive interests specified. Returning all places.")
        return df[df['rating'] >= min_rating].copy()
    
    # Filter: must have at least one positive interest
    def has_positive_interest(categories):
        return any(cat in positive_interests for cat in categories)
    
    # Filter: must not have negative interests
    def has_negative_interest(categories):
        return any(cat in negative_interests for cat in categories)
    
    filtered_df = df[
        df['categories'].apply(has_positive_interest) &
        ~df['categories'].apply(has_negative_interest) &
        (df['rating'] >= min_rating)
    ].copy()
    
    return filtered_df

# Apply filtering
filtered_places = filter_places_by_interests(
    df,
    positive_interests,
    negative_interests,
    min_rating=4.0
)

print(f"\nFiltering Results:")
print(f"  Original dataset: {len(df)} places")
print(f"  After filtering: {len(filtered_places)} places")
print(f"  Reduction: {(1 - len(filtered_places)/len(df))*100:.1f}%\n")

if len(filtered_places) > 0:
    print("Top 10 Filtered Places (by rating):")
    top_places = filtered_places.nlargest(10, 'rating')[['name', 'rating', 'user_ratings_total', 'categories']]
    for idx, row in top_places.iterrows():
        print(f"  {row['name'][:45]:45s} | Rating: {row['rating']:.1f} ({row['user_ratings_total']:5d} reviews) | {', '.join(row['categories'])}")
else:
    print("⚠ No places matched your criteria. Try adjusting preferences or lowering min_rating.")

## 5. Visualize Filtered Places on Map

In [ ]:
# Category colors for visualization
CATEGORY_COLORS = {
    'adventure': 'orange',
    'beaches': 'lightblue',
    'food': 'green',
    'historical': 'darkred',
    'nature': 'darkgreen',
    'nightlife': 'purple',
    'religious': 'blue',
    'shopping': 'pink',
    'other': 'gray'
}

def get_marker_color(categories: List[str]) -> str:
    """Get color for marker based on primary category."""
    if not categories:
        return 'gray'
    # Return color of first category
    return CATEGORY_COLORS.get(categories[0], 'gray')

def create_folium_map(
    df: pd.DataFrame,
    title: str = "Goa Tourist Attractions",
    use_clusters: bool = False
) -> folium.Map:
    """
    Create an interactive Folium map with place markers.
    
    Args:
        df: DataFrame with places (must have lat, lon, name, etc.)
        title: Map title
        use_clusters: Whether to cluster nearby markers
    
    Returns:
        Folium Map object
    """
    if len(df) == 0:
        print("No places to display on map.")
        return None
    
    # Calculate center
    center_lat = df['latitude'].mean()
    center_lon = df['longitude'].mean()
    
    # Create map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=10,
        tiles='OpenStreetMap'
    )
    
    # Add title
    title_html = f'''
    <div style="position: fixed; 
                top: 10px; left: 50px; width: 400px; height: 50px; 
                background-color: white; border: 2px solid grey; z-index: 9999;
                font-size: 16px; font-weight: bold; padding: 10px;">
        {title}<br>
        <span style="font-size: 12px; font-weight: normal;">Total Places: {len(df)}</span>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    # Add markers
    if use_clusters:
        marker_cluster = MarkerCluster().add_to(m)
    
    for idx, row in df.iterrows():
        # Create popup content
        popup_html = f"""
        <div style="width: 200px;">
            <h4>{row['name']}</h4>
            <b>Rating:</b> {row['rating']} ⭐ ({row['user_ratings_total']} reviews)<br>
            <b>Categories:</b> {', '.join(row['categories'])}<br>
            <b>Address:</b> {row['address'][:80]}...
        </div>
        """
        
        marker = folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=row['name'],
            icon=folium.Icon(
                color=get_marker_color(row['categories']),
                icon='info-sign'
            )
        )
        
        if use_clusters:
            marker.add_to(marker_cluster)
        else:
            marker.add_to(m)
    
    # Add legend
    legend_html = '''
    <div style="position: fixed; 
                bottom: 50px; right: 50px; width: 150px;
                background-color: white; border: 2px solid grey; z-index: 9999;
                font-size: 12px; padding: 10px;">
        <b>Categories:</b><br>
    '''
    
    for category, color in CATEGORY_COLORS.items():
        legend_html += f'<span style="color:{color};">⬤</span> {category.title()}<br>'
    
    legend_html += '</div>'
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m

# Create map for filtered places
if len(filtered_places) > 0:
    map_filtered = create_folium_map(
        filtered_places,
        title=f"Filtered Places (User Preferences: {', '.join(positive_interests)})",
        use_clusters=len(filtered_places) > 50
    )
    
    # Save map
    output_file = 'module1_filtered_places_map.html'
    map_filtered.save(output_file)
    print(f"\n✓ Map saved to: {output_file}")
    
    # Display in notebook
    display(map_filtered)
else:
    print("No places to visualize.")

## 6. Summary Statistics

In [ ]:
if len(filtered_places) > 0:
    print("=" * 60)
    print("MODULE 1 SUMMARY")
    print("=" * 60)
    print(f"\nUser Input:")
    print(f"  \"{USER_PREFERENCE}\"")
    print(f"\nClassified Interests:")
    print(f"  ✓ Positive: {', '.join(positive_interests) if positive_interests else 'None'}")
    print(f"  ✗ Negative: {', '.join(negative_interests) if negative_interests else 'None'}")
    print(f"\nFiltering Results:")
    print(f"  Total places in dataset: {len(df)}")
    print(f"  Places matching criteria: {len(filtered_places)}")
    print(f"  Average rating: {filtered_places['rating'].mean():.2f}")
    print(f"  Rating range: {filtered_places['rating'].min():.1f} - {filtered_places['rating'].max():.1f}")
    
    # Category breakdown
    print(f"\nCategory Breakdown:")
    filtered_categories = [cat for cats in filtered_places['categories'] for cat in cats]
    category_counts = Counter(filtered_categories)
    for category, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {category:15s}: {count:3d} places")
    
    print(f"\n✓ Map visualization saved to: module1_filtered_places_map.html")
    print("=" * 60)
else:
    print("No results to summarize.")

## 7. Compare: All Places vs Filtered Places

In [ ]:
# Create comparison map showing both datasets
print("Creating comparison visualization...\n")

# All places map
map_all = create_folium_map(
    df,
    title="All Places in Dataset",
    use_clusters=True
)
map_all.save('module1_all_places_map.html')
print("✓ All places map saved to: module1_all_places_map.html")

# Display comparison
print(f"\nComparison:")
print(f"  All places: {len(df)}")
print(f"  Filtered places: {len(filtered_places)}")
print(f"  Reduction: {(1 - len(filtered_places)/len(df))*100:.1f}%")

## 8. Export Filtered Places

Export the filtered places for use in subsequent modules.

In [ ]:
if len(filtered_places) > 0:
    # Export to CSV
    output_csv = 'module1_filtered_places.csv'
    filtered_places.to_csv(output_csv, index=False)
    print(f"✓ Filtered places exported to: {output_csv}")
    
    # Export to JSON for API use
    import json
    output_json = 'module1_filtered_places.json'
    
    places_json = filtered_places[[
        'name', 'address', 'latitude', 'longitude', 'rating', 
        'user_ratings_total', 'categories', 'place_id'
    ]].to_dict('records')
    
    # Convert categories list to string for JSON
    for place in places_json:
        place['categories'] = ', '.join(place['categories'])
    
    with open(output_json, 'w') as f:
        json.dump(places_json, f, indent=2)
    
    print(f"✓ Filtered places exported to: {output_json}")
    print(f"\n✓ Module 1 complete! Filtered places ready for Module 2 (clustering).")
else:
    print("No places to export.")

---

## Next Steps

1. **Module 2 (Clustering)**: Group the filtered places by day using K-Means clustering
2. **Module 3 (Route Optimization)**: Optimize daily routes using Genetic Algorithm
3. **Integration**: Connect with hosted NLC model for real-time classification

## Notes

- Current classification uses simple keyword matching. In production, use the fine-tuned DistilBERT model.
- Adjust `min_rating` threshold based on quality requirements
- Category mappings can be refined based on actual place data analysis